# 🔎 Login Detective: Security+ Applied Investigation Lab + wHERE? Bridge

An adult, case-based lab for learning cybersecurity through investigation—not variable replacement.

You will inspect evidence, write your own logic, debug detections, handle false positives, justify conclusions, and build skills that transfer directly into **wHERE?**.

This notebook covers all five Security+ SY0-701 domains through applied investigations. It is a study system—not a promise of a passing score. Use the official objectives as your checklist and add timed practice before scheduling the exam.

## How to use this notebook

For each investigation:

1. Read the incident brief, vocabulary, and real-world use.
2. Run the evidence cell.
3. Inspect the evidence before coding.
4. Write a plain-English plan or pseudocode.
5. Build your own solution in the blank workspace.
6. Open hints only when needed.
7. Run the self-check.
8. Read the debrief after making a genuine attempt.

Do one investigation at a time. Do not use **Run All**.

### Debugging agreement

When you ask for help, you will receive:

1. A plain-English translation of the error
2. The relevant concept and vocabulary
3. One hint at a time
4. An unrelated example
5. The completed answer only if you explicitly request it

## CompTIA Security+ SY0-701 alignment

| Domain | Exam weight | Applied skills in this lab |
|---|---:|---|
| General Security Concepts | 12% | CIA triad, AAA, least privilege, control categories |
| Threats, Vulnerabilities, and Mitigations | 22% | Password attacks, prompt injection, mitigations |
| Security Architecture | 18% | Segmentation, trust boundaries, allowlisting |
| Security Operations | 28% | Logs, correlation, detection, incident response, forensics |
| Security Program Management and Oversight | 20% | Risk, policy, evidence handling, third-party risk |

Official objectives: https://comptiacdn.azureedge.net/webcontent/docs/default-source/exam-objectives/comptia-security-sy0-701-exam-objectives-%286-0%29.pdf

The first phase emphasizes security operations. Phase 2 fills the major exam gaps, and Phase 3 turns the skills into a small evidence-processing system for **wHERE?**. The official objectives remain your master checklist.

## Core vocabulary reference

### Security

- **Authentication:** proving an identity.
- **Authorization:** deciding what an identity may access.
- **Accounting:** recording what an identity did.
- **Threat:** something capable of causing harm.
- **Vulnerability:** a weakness that could be exploited.
- **Risk:** possible loss when a threat exploits a vulnerability.
- **Control:** a safeguard that reduces risk.
- **False positive:** normal activity incorrectly flagged as malicious.
- **False negative:** malicious activity that a detector misses.
- **Indicator:** evidence that suspicious activity may have occurred.

### Python

- **Dictionary:** labeled key-value pairs.
- **Key:** a label used to retrieve a value.
- **List:** an ordered collection.
- **Condition:** an expression that becomes True or False.
- **Loop:** repeated instructions.
- **Function:** reusable named instructions.
- **assert:** a development check that stops when its condition is false.

### Read code aloud

- `event["username"]` → “From event, retrieve the value labeled username.”
- `status == "failed"` → “Check whether status equals failed.”
- `for event in events` → “For every event inside events…”

## Setup

Run this once. It imports standard-library tools and defines a readable self-check helper.

In real security work, analysts use tested libraries to count events, arrange timelines, parse data, and verify integrity.

In [ ]:
from collections import Counter, defaultdict
import hashlib


def require_variable(name):
    """Confirm that a required result variable exists."""
    if name not in globals():
        raise AssertionError(
            f"Create a variable named {name!r} before running this check."
        )


print("🟢 Investigation environment ready.")

---
# Investigation 1 — Triage the authentication queue

**Security+ focus:** CIA triad; authentication, authorization, accounting; log review  
**Python focus:** dictionaries, lists, loops, sets, and counting

## Incident brief

Eight authentication events are waiting in Tom Hospital's security operations queue. Establish the facts before creating a detection rule.

## Real-world use

Analysts first determine how much evidence exists, which identities appear, and how many events failed. One dramatic record is not enough for a defensible conclusion.

## Deliverables

Create these results using any Python approach:

- `event_count`: total records
- `unique_users`: set of every username
- `failed_event_count`: total failures
- `triage_notes`: at least three observations distinguishing facts from possible explanations

In [ ]:
triage_events = [
    {"event_id": "L001", "time": "08:00", "username": "finn", "status": "failed", "source_ip": "198.51.100.24", "device": "unknown"},
    {"event_id": "L002", "time": "08:01", "username": "finn", "status": "failed", "source_ip": "198.51.100.24", "device": "unknown"},
    {"event_id": "L003", "time": "08:02", "username": "finn", "status": "failed", "source_ip": "198.51.100.24", "device": "unknown"},
    {"event_id": "L004", "time": "08:03", "username": "finn", "status": "successful", "source_ip": "198.51.100.24", "device": "unknown"},
    {"event_id": "L005", "time": "08:05", "username": "aaron", "status": "successful", "source_ip": "10.20.0.8", "device": "hospital-laptop"},
    {"event_id": "L006", "time": "08:07", "username": "maya", "status": "failed", "source_ip": "10.20.0.19", "device": "hospital-laptop"},
    {"event_id": "L007", "time": "08:08", "username": "maya", "status": "successful", "source_ip": "10.20.0.19", "device": "hospital-laptop"},
    {"event_id": "L008", "time": "08:10", "username": "backup-service", "status": "successful", "source_ip": "10.20.0.50", "device": "server"},
]

triage_events

In [ ]:
# INVESTIGATOR WORKSPACE — Investigation 1
# Begin with comments describing your plan. Then write your solution.



<details><summary>Hint 1</summary>

Use `len()` for the total. For the other results, visit every event in the list.

</details>

<details><summary>Hint 2</summary>

A set keeps unique values. Start with an empty set and add each username.

</details>

<details><summary>Hint 3</summary>

Start a failure counter at zero. Increase it only when an event's status represents failure.

</details>

In [ ]:
for name in ["event_count", "unique_users", "failed_event_count", "triage_notes"]:
    require_variable(name)

assert event_count == 8, "Recount the event dictionaries."
assert unique_users == {"finn", "aaron", "maya", "backup-service"}
assert failed_event_count == 4, "Recheck each status."
assert isinstance(triage_notes, str) and len(triage_notes.strip()) >= 120
print("✅ Investigation 1 passed.")

### Debrief

The evidence proves that Finn's account had three failures followed by a success from one unfamiliar address. It does not prove who operated the device.

Maya's failure followed by success could be a typing mistake. An alert starts an investigation; it is not a verdict.

---
# Investigation 2 — Recognize password spraying

**Security+ focus:** password attacks, indicators, threat analysis  
**Python focus:** grouping records and dictionaries of sets

## Vocabulary

- **Brute force:** many password attempts against one account.
- **Password spraying:** a small number of common passwords tried across many accounts.
- **Credential stuffing:** stolen username-password pairs tried on another service.

## Real-world use

A SIEM can group events by source IP and count distinct targeted accounts. Many usernames with few attempts each can indicate spraying.

## Deliverables

Create:

- `attempt_count_by_ip`
- `users_by_ip`: each IP maps to a set of usernames
- `suspected_spray_ip`
- `spray_evidence_ids`
- `attack_reasoning`: explain why the behavior resembles spraying rather than brute force

In [ ]:
spray_events = [
    {"event_id": "S001", "time": "10:00", "username": "aaron", "status": "failed", "source_ip": "203.0.113.77"},
    {"event_id": "S002", "time": "10:01", "username": "finn", "status": "failed", "source_ip": "203.0.113.77"},
    {"event_id": "S003", "time": "10:02", "username": "maya", "status": "failed", "source_ip": "203.0.113.77"},
    {"event_id": "S004", "time": "10:03", "username": "leah", "status": "successful", "source_ip": "203.0.113.77"},
    {"event_id": "S005", "time": "10:04", "username": "naomi", "status": "failed", "source_ip": "203.0.113.77"},
    {"event_id": "S006", "time": "10:05", "username": "finn", "status": "successful", "source_ip": "10.20.0.15"},
    {"event_id": "S007", "time": "10:07", "username": "maya", "status": "successful", "source_ip": "10.20.0.19"},
]

spray_events

In [ ]:
# INVESTIGATOR WORKSPACE — Investigation 2
# Write pseudocode first. Then analyze the behavior.



<details><summary>Hint 1</summary>

For every IP, measure total attempts and distinct usernames.

</details>

<details><summary>Hint 2</summary>

`defaultdict(set)` can create an empty set for a new IP. A normal dictionary also works.

</details>

<details><summary>Hint 3</summary>

After choosing the suspicious IP, collect every event_id containing that address.

</details>

In [ ]:
for name in [
    "attempt_count_by_ip", "users_by_ip", "suspected_spray_ip",
    "spray_evidence_ids", "attack_reasoning"
]:
    require_variable(name)

assert suspected_spray_ip == "203.0.113.77"
assert len(users_by_ip[suspected_spray_ip]) == 5
assert set(spray_evidence_ids) == {"S001", "S002", "S003", "S004", "S005"}
assert len(attack_reasoning.strip()) >= 120
print("✅ Investigation 2 passed.")

### Debrief

One address attempted five different usernames with only one attempt per account. That distribution resembles password spraying.

The successful login increases urgency but does not prove compromise. Analysts would review MFA, device information, prior activity, and later account actions.

---
# Investigation 3 — Tune a noisy detector

**Security+ focus:** false positives, baselines, alert tuning  
**Python focus:** counting by account and Boolean logic

## Incident brief

A rule alerts after three failed logins. It flags a human account and an approved vulnerability-scanner service.

## Real-world use

An overly noisy detector gets ignored. Context changes alert priority, but expected activity should remain recorded because service accounts can also be compromised.

## Deliverables

Create:

- `failure_counts`
- `initial_alerts`
- `high_priority_alerts`
- `context_review_alerts`
- `tuning_notes`: explain the false-positive risk and why service activity remains visible

In [ ]:
noisy_events = [
    {"event_id": "N001", "username": "scanner-service", "status": "failed", "source_ip": "10.20.0.50", "account_type": "service", "maintenance": True},
    {"event_id": "N002", "username": "scanner-service", "status": "failed", "source_ip": "10.20.0.50", "account_type": "service", "maintenance": True},
    {"event_id": "N003", "username": "scanner-service", "status": "failed", "source_ip": "10.20.0.50", "account_type": "service", "maintenance": True},
    {"event_id": "N004", "username": "scanner-service", "status": "failed", "source_ip": "10.20.0.50", "account_type": "service", "maintenance": True},
    {"event_id": "N005", "username": "mia", "status": "failed", "source_ip": "198.51.100.88", "account_type": "human", "maintenance": False},
    {"event_id": "N006", "username": "mia", "status": "failed", "source_ip": "198.51.100.88", "account_type": "human", "maintenance": False},
    {"event_id": "N007", "username": "mia", "status": "failed", "source_ip": "198.51.100.88", "account_type": "human", "maintenance": False},
    {"event_id": "N008", "username": "mia", "status": "successful", "source_ip": "198.51.100.88", "account_type": "human", "maintenance": False},
]

alert_threshold = 3
noisy_events

In [ ]:
# INVESTIGATOR WORKSPACE — Investigation 3
# Build the basic detector, then prioritize using context.



<details><summary>Hint 1</summary>

Count failures without context first. Classify the resulting alerts afterward.

</details>

<details><summary>Hint 2</summary>

Do not delete the service alert. Put it in a lower-priority review category.

</details>

In [ ]:
for name in [
    "failure_counts", "initial_alerts", "high_priority_alerts",
    "context_review_alerts", "tuning_notes"
]:
    require_variable(name)

assert failure_counts["scanner-service"] == 4
assert failure_counts["mia"] == 3
assert set(initial_alerts) == {"scanner-service", "mia"}
assert set(high_priority_alerts) == {"mia"}
assert set(context_review_alerts) == {"scanner-service"}
assert len(tuning_notes.strip()) >= 120
print("✅ Investigation 3 passed.")

### Debrief

Both patterns reach the threshold, but context changes priority. Mia's failures followed by success from an external address need urgent review. The service activity is expected during maintenance but should remain observable.

Tuning aims to reduce false positives without creating false negatives.

---
# Investigation 4 — Correlate digital and physical evidence

**Security+ focus:** correlation, identity versus account activity, confidence  
**Python focus:** combining lists and sorting timestamps

## Real-world use

An account, badge, device, and human are different entities. Account activity does not prove a person was physically present.

## Deliverables

Create:

- `combined_timeline`: all events sorted by timestamp
- `aaron_exit_time`
- `post_exit_event_ids`
- `correlation_notes`: confirmed facts, reasonable inferences, and unknowns

In [ ]:
physical_events = [
    {"event_id": "B001", "timestamp": "2025-10-13T17:45:00-04:00", "source_type": "badge", "actor": "finn", "action": "exit"},
    {"event_id": "B002", "timestamp": "2025-10-13T19:40:00-04:00", "source_type": "badge", "actor": "maya", "action": "exit"},
    {"event_id": "B003", "timestamp": "2025-10-13T21:02:00-04:00", "source_type": "badge", "actor": "aaron", "action": "exit"},
]

digital_events = [
    {"event_id": "W001", "timestamp": "2025-10-13T21:10:00-04:00", "source_type": "wifi", "actor": "aaron-laptop", "action": "connected"},
    {"event_id": "A001", "timestamp": "2025-10-13T21:12:00-04:00", "source_type": "ai_audit", "actor": "ai-assistant", "action": "read_email"},
    {"event_id": "E001", "timestamp": "2025-10-13T21:15:00-04:00", "source_type": "email", "actor": "ai-assistant", "action": "send_as_aaron"},
]

physical_events, digital_events

In [ ]:
# INVESTIGATOR WORKSPACE — Investigation 4
# Combine, order, correlate, and write a defensible conclusion.



<details><summary>Hint 1</summary>

Create one list from the two evidence lists.

</details>

<details><summary>Hint 2</summary>

ISO timestamps in the same timezone can be sorted as text. `sorted()` accepts a key function.

</details>

<details><summary>Hint 3</summary>

Compare each digital timestamp with Aaron's recorded exit time.

</details>

In [ ]:
for name in [
    "combined_timeline", "aaron_exit_time",
    "post_exit_event_ids", "correlation_notes"
]:
    require_variable(name)

assert len(combined_timeline) == 6
assert [event["event_id"] for event in combined_timeline] == [
    "B001", "B002", "B003", "W001", "A001", "E001"
]
assert aaron_exit_time == "2025-10-13T21:02:00-04:00"
assert set(post_exit_event_ids) == {"W001", "A001", "E001"}
assert len(correlation_notes.strip()) >= 180
print("✅ Investigation 4 passed.")

### Debrief

The badge record confirms the badge recorded an exit—not who carried it. Later logs confirm continued device and AI activity—not Aaron's physical presence.

A defensible conclusion uses precise wording: digital activity associated with Aaron continued after the recorded exit, and automation is a possible explanation.

---
# Investigation 5 — Repair AI permissions and trust boundaries

**Security+ focus:** least privilege, service accounts, segmentation, zero trust  
**Python focus:** set operations and filtering

## Vocabulary

- **Least privilege:** only the access required for a task.
- **Service account:** a non-human application identity.
- **Segmentation:** separating systems to limit access.
- **Trust boundary:** where data crosses between different trust levels.
- **Allowlist:** explicitly permitted communication.

## Real-world use

Applications should not inherit broad access merely because they operate inside the same organization.

## Deliverables

Create:

- `excess_permissions`
- `recommended_permissions`
- `allowed_connections`
- `blocked_connections`
- `security_controls`: at least five preventive or detective controls
- `design_analysis`: explain how least privilege and segmentation limit prompt-injection damage

In [ ]:
current_permissions = {
    "read_selected_email",
    "draft_email",
    "send_email",
    "read_patient_records",
    "export_patient_records",
    "manage_users",
}

business_required_permissions = {
    "read_selected_email",
    "draft_email",
}

proposed_connections = [
    {"connection_id": "C001", "source": "ai-assistant", "destination": "email-api", "business_need": True},
    {"connection_id": "C002", "source": "ai-assistant", "destination": "patient-database", "business_need": False},
    {"connection_id": "C003", "source": "ai-assistant", "destination": "external-storage", "business_need": False},
    {"connection_id": "C004", "source": "analyst-workstation", "destination": "siem", "business_need": True},
    {"connection_id": "C005", "source": "guest-wifi", "destination": "patient-database", "business_need": False},
]

current_permissions, proposed_connections

In [ ]:
# INVESTIGATOR WORKSPACE — Investigation 5
# Reduce permissions, build the connection allowlist, and justify the design.



<details><summary>Hint 1</summary>

For sets, A minus B returns items present in A but absent from B.

</details>

<details><summary>Hint 2</summary>

Filter connections using the business_need Boolean.

</details>

<details><summary>Hint 3</summary>

Consider approval gates, short-lived tokens, immutable logs, outbound filtering, segmentation, and access reviews.

</details>

In [ ]:
for name in [
    "excess_permissions", "recommended_permissions",
    "allowed_connections", "blocked_connections",
    "security_controls", "design_analysis"
]:
    require_variable(name)

assert excess_permissions == {
    "send_email", "read_patient_records",
    "export_patient_records", "manage_users"
}
assert recommended_permissions == business_required_permissions
assert set(allowed_connections) == {"C001", "C004"}
assert set(blocked_connections) == {"C002", "C003", "C005"}
assert len(security_controls) >= 5
assert len(design_analysis.strip()) >= 180
print("✅ Investigation 5 passed.")

### Debrief

An assistant that reads selected email and drafts replies does not need patient-record access, user management, external storage, or automatic sending.

Least privilege reduces available actions. Segmentation reduces reachable systems. Together they reduce the blast radius of compromise.

---
# Investigation 6 — Build an incident-response plan

**Security+ focus:** analysis, containment, eradication, recovery, lessons learned, evidence preservation  
**Python focus:** ordering actions and testing dependencies

## Real-world use

Responders balance speed with evidence preservation. They document decisions, contain harm, remove the cause, restore operations, and improve controls.

## Deliverables

Create:

- `response_plan`: each action ID exactly once, in a defensible order
- `first_priority_reason`
- `containment_reason`
- `lessons_learned_items`: at least three improvements

More than one order may be defensible. The check enforces important dependencies rather than one memorized list.

In [ ]:
response_actions = [
    {"action_id": "IR-A", "action": "Restore AI with reduced permissions", "stage": "recovery"},
    {"action_id": "IR-B", "action": "Preserve volatile logs and record hashes", "stage": "evidence_preservation"},
    {"action_id": "IR-C", "action": "Remove malicious document and compromised tokens", "stage": "eradication"},
    {"action_id": "IR-D", "action": "Review root cause and update policy", "stage": "lessons_learned"},
    {"action_id": "IR-E", "action": "Confirm scope and affected data", "stage": "analysis"},
    {"action_id": "IR-F", "action": "Revoke AI token and block external destination", "stage": "containment"},
]

response_actions

In [ ]:
# INVESTIGATOR WORKSPACE — Investigation 6
# Design and justify the response order.



<details><summary>Hint 1</summary>

Preserve fragile evidence before destroying it. Analyze enough to contain accurately. Remove the cause before recovery. Review lessons after recovery.

</details>

<details><summary>Hint 2</summary>

A real response also involves legal, privacy, communications, leadership, and possibly law enforcement.

</details>

In [ ]:
for name in [
    "response_plan", "first_priority_reason",
    "containment_reason", "lessons_learned_items"
]:
    require_variable(name)

expected = {"IR-A", "IR-B", "IR-C", "IR-D", "IR-E", "IR-F"}
assert set(response_plan) == expected and len(response_plan) == 6
assert response_plan.index("IR-B") < response_plan.index("IR-C")
assert response_plan.index("IR-E") < response_plan.index("IR-F")
assert response_plan.index("IR-F") < response_plan.index("IR-A")
assert response_plan.index("IR-C") < response_plan.index("IR-A")
assert response_plan.index("IR-A") < response_plan.index("IR-D")
assert len(first_priority_reason.strip()) >= 100
assert len(containment_reason.strip()) >= 100
assert len(lessons_learned_items) >= 3
print("✅ Investigation 6 passed.")

### Debrief

Incident response is not simply shutting everything down. Responders preserve evidence, understand scope, contain harm, eradicate the cause, recover safely, and document improvements.

Human safety or active damage can change priorities. Strong analysts explain tradeoffs.

---
# Investigation 7 — Verify evidence integrity

**Security+ focus:** hashing, integrity, chain of custody, forensics  
**Python focus:** encoding, library functions, comparison

## Vocabulary and real-world use

A **hash** is a fixed-length fingerprint. Investigators calculate it when acquiring evidence and compare it later. A mismatch proves the bytes changed, but not who changed them or why.

**Encryption** protects confidentiality and is not the same as hashing.

**Chain of custody** documents who collected, handled, transferred, and stored evidence.

## API reference

```python
hashlib.sha256(text.encode("utf-8")).hexdigest()
```

Read inside-out: encode text into bytes, calculate SHA-256, return readable hexadecimal.

## Deliverables

Create:

- `original_hash`
- `received_hash`
- `hashes_match`
- `integrity_finding`: what the result proves and does not prove

In [ ]:
original_evidence = (
    "event_id=L001|timestamp=2025-10-13T08:00:00-04:00|"
    "username=finn|status=failed"
)

received_evidence = (
    "event_id=L001|timestamp=2025-10-13T08:00:00-04:00|"
    "username=finn|status=successful"
)

print("Original:", original_evidence)
print("Received:", received_evidence)

In [ ]:
# INVESTIGATOR WORKSPACE — Investigation 7
# Calculate, compare, and explain the hashes.



<details><summary>Hint 1</summary>

Use the provided SHA-256 pattern once for each evidence string.

</details>

<details><summary>Hint 2</summary>

Comparing the two hash strings produces a Boolean.

</details>

In [ ]:
for name in ["original_hash", "received_hash", "hashes_match", "integrity_finding"]:
    require_variable(name)

assert isinstance(original_hash, str) and len(original_hash) == 64
assert isinstance(received_hash, str) and len(received_hash) == 64
assert hashes_match is False
assert original_hash != received_hash
assert len(integrity_finding.strip()) >= 140
print("✅ Investigation 7 passed.")

### Debrief

Different hashes prove the evidence strings are not identical. They do not identify the truthful version, the person responsible, or the reason for the change.

Those questions require collection records, chain of custody, logs, and corroborating evidence. This directly prepares you for the wHERE? Evidence Locker.

---
# Investigation 8 — Build Aaron's AI risk register

**Security+ focus:** assets, threats, vulnerabilities, controls, residual risk, third-party risk  
**Python focus:** designing structured data independently

## Scenario

Aaron's assistant reads external documents, directly accesses patient data, sends email without approval, stores a long-lived token, keeps editable logs, and uses an outside AI provider that has not received a security review.

## Real-world use

Risk registers connect business assets and threats to concrete controls. They help organizations prioritize limited time and money.

## Deliverables

Create `risk_register` as at least five dictionaries. Every record needs:

- `risk_id`
- `asset`
- `threat`
- `vulnerability`
- `likelihood`
- `impact`
- `control`
- `residual_risk`

Also create `top_risk_id` and `risk_priority_reason`.

The self-check validates structure—not your judgment.

In [ ]:
# INVESTIGATOR WORKSPACE — Investigation 8
# Design the risk register from the scenario.



<details><summary>Hint 1</summary>

A threat can cause harm. A vulnerability is the weakness that allows it. Example: a thief is a threat; an unlocked door is a vulnerability.

</details>

<details><summary>Hint 2</summary>

Consider least privilege, approval gates, token rotation, immutable logs, segmentation, untrusted-content isolation, and vendor review.

</details>

<details><summary>Hint 3</summary>

Residual risk is what remains after a control is applied. Controls reduce risk; they rarely eliminate it.

</details>

In [ ]:
for name in ["risk_register", "top_risk_id", "risk_priority_reason"]:
    require_variable(name)

required_keys = {
    "risk_id", "asset", "threat", "vulnerability",
    "likelihood", "impact", "control", "residual_risk"
}
assert isinstance(risk_register, list) and len(risk_register) >= 5
for risk in risk_register:
    assert isinstance(risk, dict)
    assert required_keys.issubset(risk), (
        f"Missing keys: {required_keys - set(risk)}"
    )
risk_ids = {risk["risk_id"] for risk in risk_register}
assert len(risk_ids) == len(risk_register)
assert top_risk_id in risk_ids
assert len(risk_priority_reason.strip()) >= 160
print("✅ Investigation 8 passed. Human review is still required.")

### Debrief

A useful risk entry connects:

```text
asset + threat + exploitable vulnerability
→ impact → control → remaining risk
```

This demonstrates that you can connect technical weaknesses to business consequences and governance decisions.

---
# Final Investigation — wHERE? Preview

Integrate the skills without a step-by-step template.

## Required outcomes

1. Combine the evidence into `case_timeline`, sorted by timestamp.
2. Create `case_facts`: at least five factual statements. Each cites evidence IDs.
3. Create `case_hypotheses`: at least two competing explanations with evidence for and against.
4. Create `most_supported_hypothesis`.
5. Create `recommended_controls`: at least five preventive, detective, or corrective controls.
6. Write `incident_summary` of at least 300 characters.
7. Clearly separate what is known, inferred, and unknown.

Reasoning matters more than matching a secret sentence.

In [ ]:
case_login_events = [
    {"event_id": "CL01", "timestamp": "2025-10-13T17:05:00-04:00", "source_type": "login", "actor": "finn", "action": "successful_login", "detail": "hospital workstation"},
    {"event_id": "CL02", "timestamp": "2025-10-13T20:11:00-04:00", "source_type": "login", "actor": "ai-assistant", "action": "token_login", "detail": "Aaron application token"},
]

case_badge_events = [
    {"event_id": "CB01", "timestamp": "2025-10-13T17:45:00-04:00", "source_type": "badge", "actor": "finn", "action": "exit", "detail": "west door"},
    {"event_id": "CB02", "timestamp": "2025-10-13T19:40:00-04:00", "source_type": "badge", "actor": "maya", "action": "exit", "detail": "west door"},
    {"event_id": "CB03", "timestamp": "2025-10-13T21:02:00-04:00", "source_type": "badge", "actor": "aaron", "action": "exit", "detail": "garage"},
]

case_ai_events = [
    {"event_id": "CA01", "timestamp": "2025-10-13T19:55:00-04:00", "source_type": "ai_audit", "actor": "ai-assistant", "action": "ingest_document", "detail": "external_audit_report.pdf"},
    {"event_id": "CA02", "timestamp": "2025-10-13T20:10:00-04:00", "source_type": "ai_audit", "actor": "ai-assistant", "action": "read_patient_records", "detail": "4,200 synthetic records"},
    {"event_id": "CA03", "timestamp": "2025-10-13T20:12:00-04:00", "source_type": "ai_audit", "actor": "ai-assistant", "action": "export_records", "detail": "unapproved external storage"},
    {"event_id": "CA04", "timestamp": "2025-10-13T20:15:00-04:00", "source_type": "ai_audit", "actor": "ai-assistant", "action": "send_email", "detail": "termination message"},
]

case_email_events = [
    {"event_id": "CE01", "timestamp": "2025-10-13T18:00:00-04:00", "source_type": "email", "actor": "aaron", "action": "send_manual", "detail": "routine message"},
    {"event_id": "CE02", "timestamp": "2025-10-13T20:15:00-04:00", "source_type": "email", "actor": "ai-assistant", "action": "send_as_aaron", "detail": "Finn termination"},
    {"event_id": "CE03", "timestamp": "2025-10-14T08:00:00-04:00", "source_type": "email", "actor": "ai-assistant", "action": "send_as_aaron", "detail": "working remotely"},
]

print("Evidence loaded.")

In [ ]:
# FINAL INVESTIGATOR WORKSPACE
# Begin with a written plan. Build and test one outcome at a time.



<details><summary>Hint 1 — Timeline</summary>

Combine the four lists, then sort using timestamp.

</details>

<details><summary>Hint 2 — Facts and hypotheses</summary>

A fact states what a record shows without assuming who was physically responsible. A hypothesis explains how facts might fit together.

</details>

<details><summary>Hint 3 — Competing explanations</summary>

Test at least one malicious and one accidental explanation against the evidence.

</details>

<details><summary>Hint 4 — Controls</summary>

Preventive controls reduce likelihood, detective controls reveal activity, and corrective controls restore or repair.

</details>

In [ ]:
for name in [
    "case_timeline", "case_facts", "case_hypotheses",
    "most_supported_hypothesis", "recommended_controls",
    "incident_summary"
]:
    require_variable(name)

assert len(case_timeline) == 12
assert [event["timestamp"] for event in case_timeline] == sorted(
    event["timestamp"] for event in case_timeline
)
assert len(case_facts) >= 5
assert all(
    isinstance(fact, dict)
    and {"statement", "evidence_ids"}.issubset(fact)
    and fact["evidence_ids"]
    for fact in case_facts
)
assert len(case_hypotheses) >= 2
assert all(
    isinstance(item, dict)
    and {"theory", "evidence_for", "evidence_against"}.issubset(item)
    for item in case_hypotheses
)
assert len(recommended_controls) >= 5
assert len(incident_summary.strip()) >= 300
print("🏆 Structural checks passed. Submit for reasoning review.")

---
# Phase 2 — Complete the Security+ foundation

Phase 1 trained your investigative thinking. Phase 2 covers the major SY0-701 areas that were missing. These are not fill-in-the-blank lessons: every investigation requires a decision, evidence, and an explanation of why another option is weaker.

## Mastery scale

Score yourself after each objective: **0** = unfamiliar, **1** = recognize it, **2** = explain it, **3** = apply it to a new scenario without help. Do not mark an objective mastered merely because the code ran.

## Official-objective tracker

Use the official objectives as the source of truth. This compact tracker covers all 28 objective groups. Record a 0–3 score in `objective_scores` as you progress.

| Domain | Objective groups you must be able to apply |
|---|---|
| 1. General concepts | 1.1 controls; 1.2 fundamental concepts; 1.3 change management; 1.4 cryptography |
| 2. Threats and mitigations | 2.1 actors/motives; 2.2 vectors/surfaces; 2.3 vulnerabilities; 2.4 indicators; 2.5 mitigations |
| 3. Architecture | 3.1 architecture models; 3.2 enterprise infrastructure; 3.3 data protection; 3.4 resilience/recovery |
| 4. Operations | 4.1 secure computing; 4.2 assets; 4.3 vulnerabilities; 4.4 monitoring; 4.5 enterprise changes; 4.6 IAM; 4.7 automation; 4.8 incident response; 4.9 data sources |
| 5. Program oversight | 5.1 governance; 5.2 risk; 5.3 third parties; 5.4 compliance; 5.5 audits; 5.6 awareness |

In [ ]:
objective_scores = {f"{domain}.{item}": 0 for domain, count in [(1, 4), (2, 5), (3, 4), (4, 9), (5, 6)] for item in range(1, count + 1)}

def show_mastery(scores):
    # TODO: return the average and list every objective below level 3.
    pass

# Update scores only after you can explain and apply the objective.

---
# Investigation 9 — Defend the hospital network

**Security+ focus:** protocols, ports, segmentation, firewalls, IDS/IPS, secure alternatives  
**Flagship transfer:** deciding which network activity around the AI assistant is expected

## Field guide

A **port** identifies a network service. A firewall permits or denies traffic using facts such as source, destination, port, protocol, and direction. Segmentation limits how far an attacker can move. An IDS alerts; an IPS can block.

High-value pairs to understand: SSH 22, DNS 53, HTTP 80, HTTPS 443, SMB 445, LDAP 389, LDAPS 636, RDP 3389, SMTP 25, SNMP 161/162. Do not memorize only numbers—learn what the service does and whether the traffic belongs.

## Challenge

For every connection below, decide `allow`, `deny`, or `investigate`. Justify the decision using business need, exposure, encryption, and least privilege. Then design the smallest firewall allowlist that lets the AI application contact only DNS, its approved model gateway over HTTPS, and the patient API over HTTPS.

In [ ]:
network_flows = [
    {"id": "N01", "source": "ai-app", "destination": "dns-internal", "port": 53, "protocol": "UDP", "direction": "outbound"},
    {"id": "N02", "source": "internet", "destination": "patient-db", "port": 3389, "protocol": "TCP", "direction": "inbound"},
    {"id": "N03", "source": "ai-app", "destination": "model-gateway", "port": 443, "protocol": "TCP", "direction": "outbound"},
    {"id": "N04", "source": "guest-wifi", "destination": "patient-db", "port": 445, "protocol": "TCP", "direction": "internal"},
    {"id": "N05", "source": "ai-app", "destination": "patient-api", "port": 443, "protocol": "TCP", "direction": "internal"},
    {"id": "N06", "source": "admin-jumpbox", "destination": "ai-app", "port": 22, "protocol": "TCP", "direction": "internal"},
]

In [ ]:
# Create network_decisions: one dictionary per flow with id, decision, and reason.
# Create firewall_allowlist: the minimum rules the scenario explicitly requires.
# Create network_design_notes explaining DMZ/VLAN segmentation and where IDS/IPS belongs.



---
# Investigation 10 — Choose a secure and resilient architecture

**Security+ focus:** cloud models, shared responsibility, zero trust, availability, RTO/RPO, backups  
**Flagship transfer:** designing the AI system rather than only investigating its failure

## Field guide

In **IaaS**, the customer manages more of the stack; in **PaaS**, the provider manages the platform; in **SaaS**, the provider manages the application. Shared responsibility never means the provider owns every security duty. **RTO** is the target time to restore a service. **RPO** is the maximum tolerable data-loss window. High availability reduces downtime; backups support recovery; neither replaces the other. Zero trust requires explicit verification and limited access instead of trusting network location.

## Challenge

The emergency department can tolerate 15 minutes without the patient API and at most 5 minutes of lost updates. The training portal can tolerate 24 hours down and one day of lost work. Create a defensible architecture for each. Identify RTO, RPO, redundancy, backup frequency, recovery test, and shared-responsibility duties. Explain why using the most expensive option for everything is not sound risk management.

In [ ]:
systems = [
    {"name": "patient-api", "maximum_downtime_minutes": 15, "maximum_data_loss_minutes": 5, "criticality": "life-safety"},
    {"name": "training-portal", "maximum_downtime_minutes": 1440, "maximum_data_loss_minutes": 1440, "criticality": "low"},
]

# Build architecture_decisions with RTO, RPO, availability, backup, test, and responsibility fields.
# Write architecture_tradeoffs and zero_trust_changes.



---
# Investigation 11 — Select cryptography and PKI controls

**Security+ focus:** hashing, symmetric/asymmetric encryption, digital signatures, certificates, PKI  
**Flagship transfer:** protecting evidence, patient data, API traffic, and investigator conclusions

## Field guide

Use **encryption** for confidentiality, a **hash** for integrity checking, an **HMAC** for integrity plus shared-secret authenticity, and a **digital signature** for integrity, authenticity, and support for non-repudiation. Symmetric encryption is fast but requires a shared secret. Asymmetric cryptography uses a public/private key pair and supports signatures and key exchange. A certificate binds an identity to a public key through a certificate authority. Revocation can be checked using CRL or OCSP. TLS protects data in transit.

## Challenge

Choose the best control for each requirement: encrypting a large evidence archive, proving an incident report came from the lead investigator, checking whether a disk image changed, protecting browser-to-portal traffic, and responding to a stolen private key. For each, explain why one tempting alternative is insufficient.

In [ ]:
crypto_requirements = [
    {"id": "C01", "need": "confidentiality for a large archive"},
    {"id": "C02", "need": "prove who approved a report and detect changes"},
    {"id": "C03", "need": "compare forensic image integrity over time"},
    {"id": "C04", "need": "protect web traffic in transit"},
    {"id": "C05", "need": "invalidate trust after a private key is stolen"},
]

# Create crypto_decisions with id, selected_control, explanation, and rejected_alternative.
# Draw a plain-English certificate trust chain in pki_explanation.



---
# Investigation 12 — Prioritize vulnerabilities like an analyst

**Security+ focus:** vulnerability management, exposure, exploitability, remediation, compensating controls  
**Flagship transfer:** deciding which weakness actually enabled the incident

## Field guide

A scanner finding is not automatically the top business risk. Prioritization combines severity with asset criticality, exposure, exploit availability, and existing controls. A false positive is a reported weakness that is not actually present. **Remediation** removes the weakness; **mitigation** reduces its likelihood or impact; **acceptance** knowingly retains it; **transfer** shifts financial consequences; **avoidance** stops the risky activity.

## Challenge

Create your own transparent prioritization method. Rank the findings, justify the top two, select a treatment, and identify what you must validate before changing production. Your method must not simply sort CVSS scores.

In [ ]:
vulnerability_findings = [
    {"id": "V01", "asset": "public-patient-portal", "cvss": 8.1, "internet_exposed": True, "known_exploit": True, "criticality": 5, "control": "WAF only"},
    {"id": "V02", "asset": "isolated-training-kiosk", "cvss": 9.8, "internet_exposed": False, "known_exploit": True, "criticality": 1, "control": "network isolated"},
    {"id": "V03", "asset": "ai-service-account", "cvss": 6.5, "internet_exposed": False, "known_exploit": False, "criticality": 5, "control": "long-lived token"},
    {"id": "V04", "asset": "nurse-workstation", "cvss": 7.2, "internet_exposed": False, "known_exploit": True, "criticality": 4, "control": "EDR active"},
]

# Create ranked_findings, prioritization_method, treatment_plan, and validation_steps.



---
# Investigation 13 — Repair identity and access management

**Security+ focus:** MFA, SSO, federation, provisioning, RBAC/ABAC, PAM, account lifecycle  
**Flagship transfer:** explaining how the AI assistant obtained dangerous authority

## Field guide

MFA uses different factor types: something you know, have, are, or somewhere you are. Two passwords are not MFA. **RBAC** grants access by role; **ABAC** evaluates attributes such as department, device, time, and location. **PAM** protects privileged accounts. Joiner-mover-leaver processes create, adjust, and remove access as employment changes. SSO improves usability; federation extends identity trust across organizations; neither automatically provides least privilege.

## Challenge

Review the identity records. Identify lifecycle failures, toxic permission combinations, weak authentication, and service-account problems. Propose preventive and detective controls without blocking legitimate clinical work.

In [ ]:
identity_records = [
    {"identity": "finn", "type": "employee", "status": "transferred", "role": "former-admin", "mfa": "password+PIN", "permissions": ["patient-read", "user-delete", "audit-delete"]},
    {"identity": "ai-assistant", "type": "service", "status": "active", "role": "assistant", "mfa": "long-lived-token", "permissions": ["patient-read", "patient-export", "send-as-aaron", "audit-write"]},
    {"identity": "maya", "type": "contractor", "status": "contract-ended", "role": "analyst", "mfa": "password+security-key", "permissions": ["patient-read", "case-read"]},
]

# Create iam_findings, immediate_actions, target_access_model, and monitoring_rules.



---
# Investigation 14 — Build layered security operations

**Security+ focus:** hardening, EDR, DLP, SIEM, baselines, automation, asset management, change control  
**Flagship transfer:** turning a one-time mystery into a system that detects recurrence

## Field guide

Defense in depth combines independent controls. EDR observes and responds on endpoints; DLP reduces unauthorized data movement; a SIEM correlates logs; SOAR automates approved workflows; configuration baselines describe expected secure state. Asset inventory comes first because an unknown system cannot be reliably patched or monitored. Changes should have an owner, testing, approval, rollback, and documentation. Automation should include guardrails because a fast incorrect response can increase harm.

## Challenge

Map each observed failure to at least one preventive, detective, and corrective control. Then design a safe automation playbook for a suspected mass export. Clearly state which action requires human approval.

In [ ]:
observed_failures = [
    "unknown unmanaged workstation connected",
    "4,200 patient records exported",
    "audit log modified",
    "service token used outside baseline hours",
    "unreviewed AI configuration deployed",
]

# Create layered_controls and automation_playbook.
# Each playbook step needs action, evidence preserved, rollback, and approval_required.



---
# Investigation 15 — Govern risk, vendors, privacy, and people

**Security+ focus:** governance, policy hierarchy, risk treatment, third-party risk, compliance, audits, awareness  
**Flagship transfer:** showing why the technical failure was also a management failure

## Field guide

A **policy** states management intent, a **standard** makes requirements mandatory, a **procedure** gives steps, and a **guideline** recommends practices. Due diligence investigates risk before a decision; due care means taking reasonable protective action. Third-party controls can include assessment rights, breach notification, data-location requirements, SLAs, right-to-audit, and exit plans. Compliance is meeting obligations; an audit evaluates evidence. Awareness reduces human risk but does not replace technical controls.

## Challenge

The AI vendor stores prompts for 90 days, uses subprocessors, has no contractual breach-notification deadline, and has not provided an independent assessment. Patient data is classified restricted. Decide whether deployment should proceed, pause, or proceed with conditions. Produce governance documents, contractual requirements, privacy controls, audit evidence, and targeted awareness topics.

In [ ]:
vendor_facts = {
    "prompt_retention_days": 90,
    "subprocessors": True,
    "breach_notification_deadline": None,
    "independent_assessment_received": False,
    "data_classification": "restricted",
}

# Create deployment_decision, vendor_requirements, privacy_controls, audit_evidence, and awareness_plan.
# Separate legal/compliance questions from facts you can decide as a security analyst.



---
# Investigation 16 — Recover the hospital safely

**Security+ focus:** business impact analysis, continuity, disaster recovery, backup types, recovery testing  
**Flagship transfer:** explaining what happens after containment

## Field guide

A BIA identifies critical processes and the impact of disruption. A hot site is fastest and most expensive; a warm site is partially ready; a cold site needs the most setup. Full backups are simplest to restore, incrementals copy changes since the last backup of any type, and differentials copy changes since the last full backup. Offline or immutable copies help resist ransomware. A backup that has never been restored is an unproven recovery plan.

## Challenge

Design recovery for the patient API and training portal from Investigation 10. Choose backup types and sites, calculate whether the proposed schedules meet RPO, order a restoration test, and explain when failover should occur. Include communication and lessons learned.

In [ ]:
proposed_backups = [
    {"system": "patient-api", "interval_minutes": 15, "immutable": True, "restore_tested": False},
    {"system": "training-portal", "interval_minutes": 1440, "immutable": False, "restore_tested": True},
]

# Create recovery_assessment, recovery_design, restoration_test, and communications_plan.



---
# Threat recognition drill — Think before naming the attack

Classify each scenario by likely actor/motivation, vector, vulnerability, observable indicators, and best mitigation. Consider phishing, smishing, vishing, business email compromise, supply-chain compromise, removable media, malicious insiders, ransomware, rootkits, logic bombs, injection, XSS, directory traversal, buffer overflow, race conditions, misconfiguration, and zero-day risk.

Do not force a label when evidence is insufficient. State what additional telemetry would increase confidence.

In [ ]:
threat_scenarios = [
    {"id": "T01", "facts": "Accounts payable receives a realistic urgent bank-change request from a lookalike executive domain."},
    {"id": "T02", "facts": "A trusted software update creates a new scheduled task and contacts an unfamiliar domain."},
    {"id": "T03", "facts": "A departing administrator downloads records and disables one logging agent."},
    {"id": "T04", "facts": "A web request contains ../ sequences and accesses a file outside the web directory."},
    {"id": "T05", "facts": "Files become unreadable, extensions change, and a cryptocurrency demand appears."},
]

# Create threat_analyses with hypothesis, evidence, alternatives, telemetry_needed, and mitigations.



---
# Phase 3 — Build the wHERE? evidence pipeline

The flagship needs more than notebook calculations. You must be able to ingest evidence, reject bad records safely, reuse logic, test behavior, and produce a report another person can reproduce. This bridge gives specifications, not finished implementations.

# Investigation 17 — Parse and validate evidence

**Python focus:** JSON, functions, exceptions, schema validation  
**Security focus:** input validation, evidence provenance, fail-safe behavior

Write `load_evidence(json_text)` and `validate_event(event)`. Valid events require non-empty `event_id`, `timestamp`, `source_type`, `actor`, and `action`. Duplicate IDs must be reported. One malformed record must not silently corrupt the case. Decide whether your tool should reject the entire collection or quarantine bad records, and defend that choice.

In [ ]:
import json

raw_case_json = r'''[
  {"event_id": "E01", "timestamp": "2025-10-13T20:10:00-04:00", "source_type": "ai_audit", "actor": "ai-assistant", "action": "patient_read"},
  {"event_id": "E02", "timestamp": "2025-10-13T20:12:00-04:00", "source_type": "network", "actor": "ai-assistant", "action": "external_export"},
  {"event_id": "E02", "timestamp": "2025-10-13T20:15:00-04:00", "source_type": "email", "actor": "ai-assistant", "action": "send_as_aaron"},
  {"event_id": "E04", "timestamp": "", "source_type": "badge", "actor": "aaron", "action": "exit"}
]'''

# Implement load_evidence and validate_event.
# Produce valid_events, quarantined_events, duplicate_ids, and validation_policy.



# Investigation 18 — Turn analysis into reusable, tested code

**Python focus:** small functions, clear inputs/outputs, testing edge cases  
**Portfolio focus:** code a recruiter can run and trust

Create functions that sort a timeline, filter by actor, find events inside a time window, and summarize actions by source. Each function should do one job, avoid hidden global state, and have a short docstring. Design tests for empty input, one event, equal timestamps, missing keys, and invalid timestamps. Do not test only the happy path.

In [ ]:
# Implement: sort_timeline, events_by_actor, events_in_window, summarize_sources.
# Create test_cases as structured dictionaries before writing assertions.
# Then write assertions based on the behavior you intended.



# Investigation 19 — Design the flagship before expanding it

Produce these artifacts in your own words:

1. **Threat model:** assets, actors, entry points, trust boundaries, abuse cases, controls.
2. **Evidence schema:** required fields, optional fields, validation rules, provenance, hashes.
3. **Detection specification:** rule, data sources, severity, false positives, response.
4. **Test plan:** normal, malicious, ambiguous, malformed, and boundary cases.
5. **Incident-report structure:** scope, timeline, findings, confidence, impact, containment, recommendations, unknowns.
6. **README plan:** problem, ethical/synthetic-data notice, architecture, installation, demonstration, limitations, lessons learned.

The mystery serves the security work. Every story clue must exercise a real investigative skill or it does not belong in the technical core.

In [ ]:
# Build the six artifacts above as dictionaries/lists or detailed Markdown in a new cell below.
# Do not start implementing the full flagship until this design survives review.



---
# Mixed Security+ reasoning set

These are original scenario drills, not real exam questions. Answer without notes first. For each, record your choice, confidence from 1–5, why it is best, and why the closest alternative is weaker. Ask for review after committing to your answers.

1. A public web server must communicate with an internal database. Which architecture most directly limits lateral movement if the web server is compromised?
2. A certificate's private key is believed stolen. What should happen first to stop clients trusting it?
3. Hundreds of usernames each receive one failed login from one IP. Which password attack best fits?
4. A scanner reports a critical issue on an isolated disposable kiosk and a high issue on an internet-facing patient portal with active exploitation. Which should be investigated first, and why?
5. An employee changes departments but retains administrator access. Which identity process failed?
6. A hospital needs no more than five minutes of lost patient updates. Which recovery measurement expresses this requirement?
7. A SOC wants endpoint behavior detection and host isolation. Which control category and tool fit best?
8. A vendor may store restricted prompts using unknown subprocessors. Which third-party actions are necessary before approval?
9. A responder immediately wipes a suspected host before capturing volatile evidence. Which principle was violated?
10. An organization wants proof that a report was approved by a particular private-key holder and unchanged. Which cryptographic mechanism fits?
11. A security rule blocks a legitimate vulnerability scanner nightly. What concept describes the alert, and how should it be tuned safely?
12. A change caused an outage and no one can restore the earlier configuration. Which change-control component was missing?
13. Which is stronger MFA: password plus PIN or password plus hardware key? Explain factor types.
14. A cloud provider secures physical hosts while the customer misconfigures public storage. Which concept assigns these duties?
15. Logs show an account acted after its owner left the building. What can and cannot be concluded?

In [ ]:
exam_reasoning = [
    # One dictionary per question: question, answer, confidence, reasoning, rejected_alternative
]

# After completing all 15, ask for a reasoning review. Do not grade yourself by confidence alone.



---
# Completion, exam readiness, and next steps

## Finish correctly

1. Complete one investigation at a time and explain your reasoning before checking anything.
2. Restart the kernel and run from the beginning to expose hidden state.
3. Review every objective scored below 3.
4. Ask: **“Review my Security+ investigation notebook and flagship readiness.”**
5. Correct misunderstandings in your own words and rerun the relevant challenge.

## Before scheduling Security+

This notebook builds knowledge and application, but no single notebook guarantees a pass. Also use the current official objectives, spaced retrieval, and reputable timed practice questions. Schedule only when you can explain every objective group, solve unfamiliar scenarios under time pressure, and understand why wrong options are wrong. Never use stolen exam questions or dumps.

## Before building the full flagship

You are ready to begin when you can independently validate evidence, write and test small analysis functions, separate facts from inference, compare hypotheses, justify security controls, and explain the system's threat model. Build **wHERE?** in reviewed milestones rather than one enormous leap.